In [ ]:
# Add this at the top of your notebook
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

: 

In [ ]:
# importing libraries
import pandas as pd

## Loading the data
- Loading the melbourne data and then viewing it

In [ ]:
# save filepath to variable for easier access
filepath = f"D:\Practice Projects\Python\Machine Learning\Kaggle Machine Learning\Melbourne Housing Problem\melb_data.csv"

melbourne_data = pd.read_csv(filepath)

# read the data and store data in DataFrame titled melbourne_data
melbourne_data.head()

## Data Exploration
- Reviewing the data to understand the patterns, format, and nature of the data.

In [ ]:
# print a summary of the data in the Melbourne data
melbourne_data.describe()

In [ ]:
# showing the features in the dataset
melbourne_data.columns

In [ ]:
# dropna drops missing values (think of na as "not available")
melbourne_data = melbourne_data.dropna(axis=0)
print("Missing values dropped....")

## Selecting Data for Modeling
- Using own intuition to determine the variables to be used as features in the model.
- There are two ways of selecting a subset of data:
1. **Dot notation** - we use this approach to select the "prediction target" (column we want to predict)
2. **Selecting** - with a column list, which we use to select the "features" (columns to be inputted into our model)

In [ ]:
# using dot notation to pull out and define the prediction/target variable
# y variable is assigned to the prediction/target variable
y = melbourne_data.Price

In [ ]:
# choosing the features to be inputted in the model
melbourne_features = ['Rooms', 'Bathroom', 'Landsize','BuildingArea', 'YearBuilt','Lattitude', 'Longtitude']

In [ ]:
# X variable is assigned to the features
X = melbourne_data[melbourne_features]

In [ ]:
# reviewing the data to get statistical summary
X.describe()

In [ ]:
# show the x variable
X.head()

## Building the Model
- We use scikit-learn library to create models which is initialized as "**sklearn**".
- The steps of building a model:
1. **Define** - What type of model will it be? A decision tree? Some other type of model? Some other parameters of the model type are specified too.
2. **Fit** - Capture patterns from provided data. This is the heart of modeling.
3. **Predict** - Just what it sounds like
4. **Evaluate** - Determine how accurate the model's predictions are.



In [ ]:
# importing the sklearn library
from sklearn.tree import DecisionTreeRegressor

# Define model. Specify a number for random_state to ensure same results each run
melbourne_model = DecisionTreeRegressor(random_state=1)

# Fit model
melbourne_model.fit(X, y)

In [ ]:
# after fitting the model, we use it to make predictions
print("Making predictions for the following 5 houses:")
print(X.head())
print("The predictions are")
print(melbourne_model.predict(X.head()))

## Model Validation
- It is important to evaluate every model that we ever build.
- The most relevant/ideal measure of model quality is predictive accuracy (will the model's predictions be close to what actually happens)
* Mistake when measuring predictive accuracy: making predictions with training data and comparing them with target values in the training data. **Problem??** - Model will always make accurate predictions and then fail on new data.
- **Best approach**, is exclude some data from model-building (training) to use the spared data (validation data) in testing model accuracy.

- Predictive accuracy measured using a single metric like **MAE (Mean Absolute Error)** as broken below:

1. **Error** - error = actual - predicted
2. **Absolute Error** - taking the absolute value of each error (convert each error to a positive number)
3. **Mean Absolute Error** - taking the average of all those absolute errors 

---
**MAE = Measure of Model Quality**

---

In [ ]:
# importing the library for MAE
from sklearn.metrics import mean_absolute_error

# assigning predicted values
predicted_home_prices = melbourne_model.predict(X)

# computing MAE for the model
mean_absolute_error(y, predicted_home_prices)

## Spliting data into training and testing 
- Mistake when measuring predictive accuracy: making predictions with training data and comparing them with target values in the training data. **Problem??** - Model will always make accurate predictions and then fail on new data.
- **Best approach**, is exclude some data from model-building (training) to use the spared data (validation data) in testing model accuracy.
- To split data, we use train_test_split function in scikit-learn to breaking up the data into two pieces: training data to fit the model & testing data for model validation to calculate mean_absolute_error (MAE)


In [ ]:
# importing the train_test_split function
from sklearn.model_selection import train_test_split

# split data into training and validation data, for both features and target
# The split is based on a random number generator. Supplying a numeric value to
# the random_state argument guarantees we get the same split every time we
# run this script.
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=0)

# defining the model 
melbourne_model = DecisionTreeRegressor()

# Fit the model
melbourne_model.fit(train_X, train_y)

# getting the predicted prices on the validation data
val_predictions = melbourne_model.predict(val_X)
print(mean_absolute_error(val_y, val_predictions))

- The above MAE is not good, we can improve the model in different ways, such as experimenting to find better features or different model types.

---

## Underfitting and Overfitting 
- To improve the performance of a model, we have to fine-tune it through identifying the best features to model.
- Firstly, we experiment on alternative models and see which model gives the best predictions. 
- From decision tree above, we keep splitting the data and getting deeper into leaves with fewer houses - leading to overfitting. 
- On the flips9de, if we only create shallow splits, we end up with leave with many houses - leading to underfitting. 

A. **Overfitting**: capturing spurious patterns that won't recur in the future, leading to less accurate predictions. Model is complex in this case since it works great on trained data, but fails on new data. 

B. **Underfitting**: failing to capture relevant patterns, again leading to less accurate predictions. Model is too simple that it performs poorly even on the data it was trained on. 

A good model should learn the correct rules so that it can answer and pass on new data it has not seen before. 


- Finding the sweet spot between underfitting and overfitting to improve accuracy while also finding the optimal tree depth.
- There are several ways of controlling tree depth, most clear and popular is the max_leaf_nodes function (sensible for controlling overfitting vs underfitting). 



In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor

def get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y):
    model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=0)
    model.fit(train_X, train_y)
    preds_val = model.predict(val_X)
    mae = mean_absolute_error(val_y, preds_val)
    return(mae)

In [ ]:
# comparing the accuracy of model built using different values for max_leaf_nodes
candidate_max_leaf_nodes = [5, 50, 500, 1000, 5000, 10000, 50000]

for max_leaf_nodes in candidate_max_leaf_nodes:
    my_mae = get_mae(max_leaf_nodes, train_X, val_X, train_y, val_y)
    print("Max leaf nodes: %d  \t\t Mean Absolute Error:  %d" %(max_leaf_nodes, my_mae))


In [ ]:
# shorter version of the above loop using a dict comprehension
scores = {leaf_size: get_mae(leaf_size, train_X, val_X, train_y, val_y) for leaf_size in candidate_max_leaf_nodes}

best_tree_size = min(scores, key=scores.get)
print(f"the best tree size {best_tree_size}")


---
- Of the options listed, 500 is the optimal number of leaves.
---

## Using a sophisticated model - Random Forests
- Most sophisticated models also face tension between overfitting and underfitting; having cleverer and different ideas to improve performance.

- **Random Forest**: Advanced that decision trees, it uses many trees and makes a prediction through averaging the predictions of each component trees. The model has better predictive accuracy compared to decision trees. 

In [ ]:
# random forest model
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

forest_model = RandomForestRegressor(random_state=1)
forest_model.fit(train_X, train_y)
melb_preds = forest_model.predict(val_X)
print(mean_absolute_error(val_y, melb_preds))